# CoMLRL MAGRPO: Cooperative TL;DR Summarization

Two Qwen2.5-0.5B agents learn to cooperate on TL;DR summarization using Multi-Agent GRPO.

- **Agent 1**: writes concise summaries (~220 chars)
- **Agent 2**: writes detailed summaries (2-3x longer)
- **Reward**: length ratio between the two responses

Optimized for **Kaggle T4 GPU (16 GB VRAM)** — uses bf16 mixed precision and minimal generation settings.

## Datasets Used in CoMLRL

The project uses three datasets; this notebook defaults to **TL;DR**.

| # | Dataset | Source | Size | Columns | Used in |
|---|---------|--------|------|---------|--------|
| 1 | **TL;DR** | HF Hub `trl-lib/tldr` | ~117k Reddit posts | `prompt` (str) | `examples/tldr-len-ratio.py` |
| 2 | **Story prompts** | `Dataset.from_dict()` (in-memory) | 32 synthetic prompts | `prompt` (str) | `examples/story-len-ratio.py` |
| 3 | **Coding tasks** | `Dataset.from_dict()` (in-memory) | 4 programming problems | `question`, `input`, `output` | `examples/leetcode-func-print.py` |

### Dataset 1: TL;DR (`trl-lib/tldr`)
```python
from datasets import load_dataset
raw = load_dataset("trl-lib/tldr", split="train")
# raw.column_names -> ['prompt']
# raw[0] -> {'prompt': 'SUBREDDIT: r/AskReddit\nTITLE: ...\nPOST: ...\nTL;DR:'}
```
Each `prompt` is a Reddit post ending with `TL;DR:` for the model to complete.

### Dataset 2: Story prompts (synthetic)
```python
from datasets import Dataset
data = {"prompt": ["Write a story about a robot:", "Explain quantum physics:", ...]}
ds = Dataset.from_dict(data)
```

### Dataset 3: Coding tasks (synthetic)
```python
from datasets import Dataset
data = {
    "question": ["Create a function that checks if ... palindrome ...", ...],
    "input": ["'radar', 'hello', 'lol'", ...],  # test inputs
    "output": ["True, False, True", ...],       # expected outputs
}
ds = Dataset.from_dict(data)
```

> **Kaggle note**: Dataset 1 auto-downloads from HF Hub (~50 MB). Datasets 2 & 3 need no download.
> You can also upload datasets as [Kaggle Datasets](https://www.kaggle.com/datasets) and read them locally to skip re-downloading.

## 1. Install Dependencies

In [1]:
!pip install -q torch>=2.0.0 transformers>=4.30.0 datasets>=2.0.0 accelerate>=0.26.0
!pip install -q git+https://github.com/OpenMLRL/CoMLRL.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Imports and Setup

In [2]:
import math
import statistics
from functools import partial

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from comlrl.trainers.reinforce import MAGRPOConfig, MAGRPOTrainer

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    # mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    # print(f"GPU: {gpu_name} ({mem_gb:.1f} GB)")

PyTorch 2.10.0+cu128  |  CUDA: True


## 3. Load, Validate, and Split Dataset

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET_SIZE = 200  # small for quick demo; increase for better results
EVAL_SPLIT = 0.2     # fraction held out for evaluation

# -- load raw dataset ---------------------------------------------------------
print("Loading TL;DR dataset...")
raw = load_dataset("trl-lib/tldr", split="train")
print(f"  Raw size: {len(raw)}")
print(f"  Columns:  {raw.column_names}")

# -- validate: inspect first few samples -------------------------------------
print("\n--- First 3 samples ---")
for i, sample in enumerate(raw.select(range(min(3, len(raw))))):
    prompt = sample.get("prompt")
    print(f"  [{i}] prompt ({len(prompt) if prompt else 0} chars): {repr(prompt[:120]) if prompt else 'MISSING'}...")

# -- validate: required fields ------------------------------------------------
if "prompt" not in raw.column_names:
    raise KeyError(f"Expected 'prompt' column; got {raw.column_names}")

# -- validate: filter bad samples ---------------------------------------------
n_before = len(raw)

def is_valid(example):
    prompt = example.get("prompt")
    if prompt is None:
        return False
    prompt = str(prompt).strip()
    if len(prompt) < 10:
        return False
    return True

raw = raw.filter(is_valid)
n_removed = n_before - len(raw)
if n_removed:
    print(f"\n  Filtered out {n_removed} invalid samples (empty/missing/too-short prompt)")
else:
    print(f"\n  All {n_before} samples pass basic validation")

# -- validate: prompt length statistics ---------------------------------------
prompt_lens = [len(str(s["prompt"]).strip()) for s in raw]
print(f"\n--- Prompt length stats (chars) ---")
print(f"  Count:  {len(prompt_lens)}")
print(f"  Min:    {min(prompt_lens)}")
print(f"  Max:    {max(prompt_lens)}")
print(f"  Mean:   {sum(prompt_lens) / len(prompt_lens):.0f}")
print(f"  Median: {sorted(prompt_lens)[len(prompt_lens)//2]}")

# -- truncate to requested size, then split train/eval -----------------------
usable = min(DATASET_SIZE, len(raw))
subset = raw.select(range(usable))

n_eval = max(1, int(usable * EVAL_SPLIT))
n_train = usable - n_eval
train_dataset = subset.select(range(n_train))
eval_dataset = subset.select(range(n_train, usable))

print(f"\n--- Train / Eval split ---")
print(f"  Train: {len(train_dataset)} prompts")
print(f"  Eval:  {len(eval_dataset)} prompts")

Loading TL;DR dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.11M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.21M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/6447 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6553 [00:00<?, ? examples/s]

  Raw size: 116722
  Columns:  ['prompt', 'completion']

--- First 3 samples ---
  [0] prompt (1864 chars): 'SUBREDDIT: r/relationships\n\nTITLE: I (f/22) have to figure out if I want to still know these girls or not and would hate'...
  [1] prompt (1135 chars): 'SUBREDDIT: r/loseit\n\nTITLE: SV & NSV! Keeping on keeping on.\n\nPOST: 30F, 5\'6". SW: 236 GW: 150 CW: 219\n\nI weigh myself w'...
  [2] prompt (1193 chars): 'SUBREDDIT: r/relationships\n\nTITLE: Me [19F] with my friend [19M] 10 months, Insecurities - Show or Tell?\n\nPOST: What are'...


Filter:   0%|          | 0/116722 [00:00<?, ? examples/s]


  All 116722 samples pass basic validation

--- Prompt length stats (chars) ---
  Count:  116722
  Min:    73
  Max:    2468
  Mean:   1406
  Median: 1406

--- Train / Eval split ---
  Train: 160 prompts
  Eval:  40 prompts


## 4. Load Tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Quick smoke test: tokenize a few prompts to verify tokenizer works
for i, sample in enumerate(train_dataset.select(range(min(3, len(train_dataset))))):
    tokens = tokenizer.encode(sample["prompt"], truncation=True, max_length=512)
    print(f"  [{i}] prompt token count: {len(tokens)}")
print("Tokenizer ready.")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  [0] prompt token count: 427
  [1] prompt token count: 307
  [2] prompt token count: 280
Tokenizer ready.


## 5. Define Reward Function, Formatters, and Eval Callbacks

In [5]:
# -- reward function ----------------------------------------------------------
def length_ratio_reward(completions1, completions2, target_min=2.0, target_max=3.0):
    """Reward agents when the second response is 2-3x longer than the first."""
    rewards = []
    for c1, c2 in zip(completions1, completions2):
        len1, len2 = len(c1), len(c2)
        if len1 == 0:
            rewards.append(0.0)
            continue
        ratio = len2 / len1
        if target_min <= ratio <= target_max:
            reward = 1.0
        elif ratio < target_min:
            reward = math.exp(-(target_min - ratio))
        else:
            reward = math.exp(-(ratio - target_max))
        rewards.append(float(reward))
    return rewards


# -- prompt formatters --------------------------------------------------------
def build_prompt_formatters(tokenizer):
    """Agent 0 = concise summarizer, Agent 1 = detailed summarizer."""
    concise_sys = "You summarize Reddit posts into concise TL;DRs (~220 characters)."
    detailed_sys = (
        "You summarize Reddit posts into detailed TL;DRs about 2-3x longer than a"
        " standard version."
    )

    formatters = []
    for sys in (concise_sys, detailed_sys):
        def _fmt(example, sys=sys):
            prompt = example.get("prompt")
            if prompt is None:
                raise KeyError("Expected 'prompt' field in dataset example.")
            apply_template = getattr(tokenizer, "apply_chat_template", None)
            if callable(apply_template):
                messages = [
                    {"role": "system", "content": sys},
                    {"role": "user", "content": prompt},
                ]
                return apply_template(messages, tokenize=False, add_generation_prompt=True)
            return f"{sys}\n\n{prompt}"
        formatters.append(_fmt)

    return formatters


# -- eval callbacks (called every eval_interval steps) -----------------------
# eval_logger: inspects each eval sample and returns per-sample metrics
def eval_logger(*, agent_completions_turns, test_cases, entry_points, prompts, **kwargs):
    """Log per-sample eval metrics: response lengths, ratio, reward."""
    samples = []
    n_agents = len(agent_completions_turns)
    # agent_completions_turns: List[List[List[str]]] — [agent][sample][turn]
    n_samples = len(agent_completions_turns[0]) if agent_completions_turns else 0

    for s_idx in range(n_samples):
        # Last-turn completions for each agent
        c1 = agent_completions_turns[0][s_idx][-1] if agent_completions_turns[0][s_idx] else ""
        c2 = agent_completions_turns[1][s_idx][-1] if agent_completions_turns[1][s_idx] else ""
        len1, len2 = len(c1), len(c2)
        ratio = len2 / max(len1, 1)
        reward = length_ratio_reward([c1], [c2])[0]
        samples.append({
            "agent_0_len": len1,
            "agent_1_len": len2,
            "ratio": ratio,
            "reward": reward,
        })

    # Print a few eval completions for inspection
    for s_idx in range(min(2, n_samples)):
        c1 = agent_completions_turns[0][s_idx][-1] if agent_completions_turns[0][s_idx] else ""
        c2 = agent_completions_turns[1][s_idx][-1] if agent_completions_turns[1][s_idx] else ""
        print(f"\n  [eval {s_idx}] Agent 0 ({len(c1)} chars): {c1[:100]}...")
        print(f"  [eval {s_idx}] Agent 1 ({len(c2)} chars): {c2[:100]}...")
        print(f"  [eval {s_idx}] Ratio: {len(c2)/max(len(c1),1):.1f}  |  Reward: {samples[s_idx]['reward']:.3f}")

    return {"samples": samples}


# eval_aggregator: aggregates per-sample metrics into scalar summaries
def eval_aggregator(detailed_metrics, *, num_turns, **kwargs):
    """Aggregate eval metrics across all samples."""
    samples = detailed_metrics.get("samples", [])
    if not samples:
        return {}

    agent0_lens = [s["agent_0_len"] for s in samples]
    agent1_lens = [s["agent_1_len"] for s in samples]
    ratios = [s["ratio"] for s in samples]
    rewards = [s["reward"] for s in samples]

    return {
        "agent_0_len_mean": float(statistics.mean(agent0_lens)),
        "agent_1_len_mean": float(statistics.mean(agent1_lens)),
        "ratio_mean": float(statistics.mean(ratios)),
        "reward_mean": float(statistics.mean(rewards)),
    }


print("Reward function, formatters, and eval callbacks ready.")

Reward function, formatters, and eval callbacks ready.


## 6. Configure and Create MAGRPO Trainer

Key Kaggle/T4 optimizations:
- `torch_dtype=bfloat16` — halves model memory vs fp32
- `num_generations=2` — minimum for GRPO, reduces KV-cache
- `max_new_tokens=128` — shorter completions = less memory
- `num_turns=1` — single-turn, no tree expansion
- `rollout_buffer_size=2` — frequent small updates
- `wandb_config=None` — no remote logging needed

### Multi-GPU support

Set `parallel_training="mp"` + `agent_devices` to run agents on separate GPUs
concurrently via `ThreadPoolExecutor`.  On Kaggle with 2×T4:

```python
config = MAGRPOConfig(
    ...,
    parallel_training="mp",
    agent_devices=["cuda:0", "cuda:1"],  # one agent per GPU
)
```

If `agent_devices` is omitted or `"auto"`, the trainer auto-discovers available GPUs.

In [6]:
# Memory-optimized configuration for T4 16GB
config = MAGRPOConfig(
    num_train_epochs=3,
    agent_learning_rate=5e-5,
    rollout_buffer_size=2,
    num_generations=2,
    max_new_tokens=64,
    temperature=0.6,
    top_p=0.6,
    num_agents=2,
    num_turns=1,
    logging_steps=10,
    eval_interval=16,      # run inline eval every 16 training steps
    eval_num_samples=1,     # number of eval prompts to run per interval
    # --- multi-GPU (uncomment on 2×T4 to put each agent on its own GPU) ---
    parallel_training="mp",
    agent_devices=["cuda:0", "cuda:1"],
)

# Load models in bf16 (or fp16 if T4 doesn't support bf16)
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    dtype = torch.bfloat16
else:
    dtype = torch.float16
model_config = {"torch_dtype": dtype}
print(f"Model dtype: {dtype}")

# Optionally enable wandb logging
# wandb_config = {"project": "comlrl-kaggle", "name": "magrpo-tldr"}
wandb_config = None  # no wandb login needed

reward_fn = partial(length_ratio_reward, target_min=2.0, target_max=3.0)

print("Creating MAGRPOTrainer...")
trainer = MAGRPOTrainer(
    agent_model=MODEL_NAME,
    model_config=model_config,
    tokenizer=tokenizer,
    reward_func=reward_fn,
    formatters=build_prompt_formatters(tokenizer),
    args=config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,       # <-- required for eval to run
    eval_logger=eval_logger,         # <-- per-sample logging
    eval_aggregator=eval_aggregator, # <-- aggregate metrics across samples
    wandb_config=wandb_config,
)
print("Trainer ready!")

`torch_dtype` is deprecated! Use `dtype` instead!


Model dtype: torch.bfloat16
Creating MAGRPOTrainer...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainer ready!


## 7. Train

Training runs with inline evaluation every `eval_interval` steps.
Watch for `eval/` prefixed metrics in the output.

In [7]:
trainer.train()

Epoch 1/3:   0%|          | 0/160 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



  [eval 0] Agent 0 (241 chars): Sure, I can help with that! Let's break down your questions and address them one by one.

### Questi...
  [eval 0] Agent 1 (280 chars): SUBREDDIT: r/college

TITLE: Did I screw up my FAFSA by supplying half my parents joint income for m...
  [eval 0] Ratio: 1.2  |  Reward: 0.432


Epoch 1/3:   1%|          | 1/160 [00:10<27:10, 10.25s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:   1%|▏         | 2/160 [00:16<21:13,  8.06s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:   2%|▏         | 3/160 [00:20<16:09,  6.18s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:   2%|▎         | 4/160 [00:26<15:50,  6.09s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:   3%|▎         | 5/160 [00:30<13:49,  5.35s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for o


  [eval 0] Agent 0 (271 chars): SUBREDDIT: r/college

TITLE: Your family's financial situation

POST: I've been trying to help your ...
  [eval 0] Agent 1 (283 chars): Sure, here's your answer:

If you want to know if you want to be answered, you can decide whether to...
  [eval 0] Ratio: 1.0  |  Reward: 0.385


Epoch 1/3:  11%|█         | 17/160 [01:38<15:05,  6.33s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:  11%|█▏        | 18/160 [01:45<15:40,  6.62s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:  12%|█▏        | 19/160 [01:49<13:40,  5.82s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:  12%|█▎        | 20/160 [01:55<13:58,  5.99s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 1/3:  13%|█▎        | 21/160 [01:59<12:36,  5.44s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 

...

Epoch 3/3:  71%|███████   | 113/160 [10:34<04:53,  6.24s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  71%|███████▏  | 114/160 [10:42<05:00,  6.52s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  72%|███████▏  | 115/160 [10:46<04:20,  5.79s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  72%|███████▎  | 116/160 [10:52<04:23,  6.00s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  73%|███████▎  | 117/160 [10:56<03:54,  5.45s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:15


  [eval 0] Agent 0 (288 chars): ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
i...
  [eval 0] Agent 1 (319 chars): Here here here here here here here here here here here here here here here here here here here here ...
  [eval 0] Ratio: 1.1  |  Reward: 0.410


Epoch 3/3:  81%|████████  | 129/160 [12:02<03:10,  6.14s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  81%|████████▏ | 130/160 [12:09<03:12,  6.41s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  82%|████████▏ | 131/160 [12:13<02:43,  5.63s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  82%|████████▎ | 132/160 [12:20<02:44,  5.89s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  83%|████████▎ | 133/160 [12:24<02:23,  5.30s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:15


  [eval 0] Agent 0 (288 chars): ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
i...
  [eval 0] Agent 1 (319 chars): Here here here here here here here here here here here here here here here here here here here here ...
  [eval 0] Ratio: 1.1  |  Reward: 0.410


Epoch 3/3:  91%|█████████ | 145/160 [13:28<01:30,  6.02s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  91%|█████████▏| 146/160 [13:36<01:30,  6.45s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  92%|█████████▏| 147/160 [13:39<01:13,  5.69s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  92%|█████████▎| 148/160 [13:46<01:11,  5.96s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Epoch 3/3:  93%|█████████▎| 149/160 [13:50<00:58,  5.34s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:15

## 8. Standalone Evaluation — Full Eval Set (Post-Training)

Run a dedicated evaluation pass over the **entire** eval dataset to get final metrics.

> **Note**: `trainer.evaluate()` accepts `num_eval_samples=` to cap the number of samples.
> Pass `len(eval_dataset)` (or a very large number) to evaluate on all eval prompts.

In [8]:
print(f"Running standalone evaluation on full eval set ({len(eval_dataset)} prompts)...")
final_metrics = trainer.evaluate(num_eval_samples=len(eval_dataset))
print(f"\nFinal eval metrics (full set): {final_metrics}")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Running standalone evaluation on full eval set (40 prompts)...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for


  [eval 0] Agent 0 (288 chars): ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
i...
  [eval 0] Agent 1 (319 chars): Here here here here here here here here here here here here here here here here here here here here ...
  [eval 0] Ratio: 1.1  |  Reward: 0.410

  [eval 1] Agent 0 (288 chars): ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
ibraltar
i...
  [eval 1] Agent 1 (319 chars): Here here here here here here here here here here here here here here here here here here here here ...
  [eval 1] Ratio: 1.1  |  Reward: 0.410

Final eval metrics (full set): {'eval/agent_0_len_mean': 288.7, 'eval/agent_1_len_mean': 319.0, 'eval/ratio_mean': 1.1051852584388184, 'eval/reward_mean': 0.4087298253096664, 'eval/turn_1/mean_reward': 0.4087298253096664, 'eval/turn_1/mean_return': 0.4087298253096665}


## 9. Save Model and Wrap Up

In [9]:
OUTPUT_DIR = "/kaggle/working/magrpo_tldr"
trainer.save_model(OUTPUT_DIR)
print(f"Models saved to {OUTPUT_DIR}")

if torch.cuda.is_available():
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_reserved(0) / 1e9:.2f} GB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Models saved to /kaggle/working/magrpo_tldr

Peak GPU memory: 8.62 GB


## Notes

- First run downloads Qwen2.5-0.5B (~1 GB) and the TL;DR dataset — takes 2-3 minutes.
- On a T4 with 200 prompts and 3 epochs, training finishes in ~30-60 minutes.
- To persist models across Kaggle sessions, save them to a Kaggle Dataset.
- For better results: increase `DATASET_SIZE` and `num_train_epochs`.
- For other reward models or environments, see `docs/content/docs/env/`.